# 후속 실험: 조용한 손해 재현과 zero-shot 비교 신호 검증

런타임 > 런타임 유형 변경 > GPU (ViT-L이라 A100 권장)

- 1단계(재현): ViT-L/14 x {ETA, SAR} x defocus_blur x seed {0,1,2} x 학습률 {1e-3, 5e-4, 2.5e-4} = 18개 실행
- 2~4단계: 1단계 결과를 분석만 한다 (GPU 불필요)
- 결과는 실행마다 드라이브에 바로 저장한다. 런타임이 끊기면 위에서부터 다시 실행하면 끝난 조합은 건너뛴다

In [ ]:
# 매번 최신 코드를 받는다 (data/는 그대로 둔다)
import os
if not os.path.isdir('/content/repo'):
    !git clone -q -b 'hyunjin_expriments#2' https://github.com/hazigamja-capstone/Milestone3_motivatingExample.git /content/repo
else:
    !git -C /content/repo pull -q
%cd /content/repo
!pip -q install open_clip_torch gdown

## 1. 데이터: ImageNet-C defocus_blur 하나

이전 실험에서 드라이브에 저장한 캐시가 있으면 그대로 가져온다. 없으면 defocus_blur만 새로 만든다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_CACHE = '/content/drive/MyDrive/motivating_cache/imagenet_c'
LOCAL = 'data/imagenet_c'
OUT = '/content/drive/MyDrive/motivating_followup'   # 결과를 드라이브에 바로 저장
os.makedirs(LOCAL, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
if os.path.exists(f'{DRIVE_CACHE}/defocus_blur_5_x.npy'):
    !cp -n {DRIVE_CACHE}/defocus_blur_5_*.npy {LOCAL}/
!python prepare_inc.py --corruptions defocus_blur --out {LOCAL}
!mkdir -p {DRIVE_CACHE} && cp -n {LOCAL}/defocus_blur_5_*.npy {DRIVE_CACHE}/
!ls -la {LOCAL}

## 2. 1단계 실행

- 학습률 1e-3부터 돌린다. 1단계 판정은 1e-3의 ETA seed 3개로 하므로, 시간이 모자라면 첫 셀만 돌리고 분석해도 된다
- 실행 하나에 ViT-L 기준 이전 실험(손상 6개)의 약 1/6 시간이 든다

In [ ]:
# 먼저: 학습률 1e-3 (ETA, SAR x seed 3 = 6개)
!python sweep.py --lrs 1e-3 --out {OUT}

In [ ]:
# 나머지 학습률 (12개)
!python sweep.py --lrs 5e-4,2.5e-4 --out {OUT}

## 3. 분석 (1~4단계)

- 1단계를 통과하지 못하면 거기서 멈춘다. 그래도 나머지를 보고 싶으면 `--force`를 붙인다
- 로그는 `OUT/followup_log.txt`, 표는 `OUT/followup_*.csv`에 저장된다

In [ ]:
!python followup.py --out {OUT} | tee {OUT}/followup_log.txt
from IPython.display import Image
Image(f'{OUT}/followup_delta.png')